<a href="https://colab.research.google.com/github/girish17work/Data-Analysis/blob/main/Unknown_Dataset_1/Unknown_Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np

In [8]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=1OtwemNuw0qilXAutyuZEjOscZVKuD7Tv'
df = pd.read_csv(DATASET_URL)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,0.087472,0.002829,1.768235,0.188396,0.117071,0.148148,0.478165,0.720443,0.967195,0.281563,0.428571,0.959596,1.0,0.0,23.358097
1,0.378379,0.022079,1.115629,0.091974,0.066089,0.851852,0.911759,0.785321,0.885001,0.424648,0.285714,0.868687,0.0,1.0,17.268768
2,0.066901,0.003828,-0.536262,0.221188,0.255671,0.296296,0.228024,0.406472,0.980184,0.274376,0.428571,0.767677,0.0,1.0,27.776974
3,0.140645,0.011132,1.323366,0.422514,0.153103,0.148148,0.410679,0.200319,0.861371,0.305006,0.142857,0.848485,0.0,1.0,16.121960
4,0.144225,0.204918,-0.930790,0.148694,0.177490,0.259259,0.146832,0.111429,0.983448,0.286322,0.285714,0.616162,1.0,0.0,23.129426


In [20]:
from sklearn.model_selection import train_test_split
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [15]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

model = Ridge(alpha=10, solver='saga', tol=1e-4, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(round(r2_score(y_test, y_pred), 3))

0.661


In [17]:
importances = np.abs(model.coef_)
most_important_feature_idx = np.argmax(importances)
print(f"The index of the most important feature is {most_important_feature_idx}")

The index of the most important feature is 9


In [18]:
least_important_feature_index = np.argmin(importances)
print(f"The index of the least important feature is {least_important_feature_index}")

The index of the least important feature is 0


In [22]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

model = SGDRegressor(random_state=42)

param_grid = {
    'penalty': ['l1', 'l2'],
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'tol': [1e-4, 1e-3, 1e-2, 1e-1]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error'
)

grid_search.fit(X_train, y_train)
print(f"Best Penalty: {grid_search.best_params_['penalty']}")

Best Penalty: l2


In [23]:
from sklearn.metrics import mean_absolute_error

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Mean Absolute Error on the test dataset: {round(mean_absolute_error(y_test, y_pred), 3)}")

Mean Absolute Error on the test dataset: 3.813


In [26]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

pipeline = Pipeline([('pca', PCA()), ('lasso', Lasso())])

param_grid = {
    'pca__n_components': [0.9, 0.95],
    'lasso__alpha': [10, 0.01, 0.001]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

y_pred = grid_search.best_estimator_.predict(X_test)
print(f"Test R2 Score: {round(r2_score(y_test, y_pred), 3)}")

Test R2 Score: 0.629


In [28]:
best_pca = grid_search.best_estimator_.named_steps['pca']
print(f"Explained Variance by the First Principal Component: {round(best_pca.explained_variance_[0], 3)}")

Explained Variance by the First Principal Component: 1.164


In [32]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

pipeline = Pipeline([('polynomial', PolynomialFeatures(degree=2, interaction_only=False)), ('lasso', Lasso(alpha=1, warm_start=True, random_state=0))])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"R2 Score on the Test Set: {round(r2_score(y_test, y_pred), 3)}")

R2 Score on the Test Set: 0.158


In [34]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

estimator = LinearRegression()
n_features = X_train.shape[1]

rfe = RFE(estimator=estimator, n_features_to_select=n_features-1)
rfe.fit(X_train, y_train)

print(f"Eliminated Index: {np.where(rfe.support_==False)[0][0]}")

Eliminated Index: 2
